# 01 · EDA: hypotheses, data quality, target and baseline

**Owner:** Machine learning · **Phase:** 1 → 2

**Project question:** can we rank Boston food establishments by their risk of a critical violation at their next
routine inspection, so inspectors visit the riskiest places first?

This notebook is organised as a set of questions and hypotheses. Each section states the hypothesis, shows the
evidence, and ends with an **Answer** cell. The summary below collects the answers; the numbers come from the
2026-10-13 landing snapshot and will move slightly as new data lands.

### Summary of answers

| # | Question / hypothesis | Answer | What we do about it |
|---|---|---|---|
| 2 | Is the raw feed clean enough? | **Mostly.** 905k violation rows and no exact duplicates, but it's one row per violation, ~10% of records are non-inspection result codes, timestamp formats are mixed, and the star levels changed meaning in 2019 | Cleaning table in section 2 → dbt staging |
| 3 | Is the inspections feed still updated? | **Yes.** Latest record is days old, volume steady | Safe to build on |
| 6 | Is "any `***` violation at a routine inspection" a usable target? | **Yes.** 21.7% base rate since 2020 (1 in 5). Follow-ups separate cleanly (96% come within 30 days of a fail) | 30-day follow-up window; handle the 2019 code change with an era flag |
| 7 | H1 Repeat offenders | **Strongly supported.** 42% after a critical vs 17% after a clean inspection | Core history features |
| 7 | H2 License category | **Supported.** Full-service 26% → take-out 20% → retail 15% → mobile 8% | `licensecat` feature |
| 7 | H3 Neighborhood | **Supported in raw rates** (11–30%), but adds almost nothing once history is in the model | Keep for now; replace with spatial join |
| 7 | H4 Seasonality | **Rejected.** Monthly rates stay within 19–24%, no summer peak | `month` is a drop candidate |
| 7 | H5 Time since last routine | **Weak, non-monotonic.** Long gaps (2+ yrs) have *half* the rate | Keep; it encodes the city's own scheduling |
| 7 | H6 New establishments are riskier | **Rejected; the reverse holds.** First-ever inspections: 10% vs 22% | `n_prior_insp = 0` captures it |
| 9 | Do licenses add useful attributes? | **Not for training, but useful for scoring.** No license number, and it's a snapshot of active licenses only, so its fields leak the future | Use as the scoring population only |
| 10 | Nearby 311 food complaints raise risk | **Promising but can't be trained on yet.** Rate rises 15% → 30% with complaint count, but 311 only covers 2026 | Backfill 311 history to 2020+ |
| 11 | Can we beat a rule of thumb? | **Yes.** Gradient boosting: valid ROC AUC 0.74, top-20% precision 44% (2.2× base rate) vs 0.63 / 41% for the best rule | Baseline to beat in notebook 02 |

**Before running:** `gcloud auth application-default login` with an account that can read the bucket, and set
`GOOGLE_CLOUD_PROJECT=ba882-team10-inspections` (otherwise gcsfs silently falls back to anonymous access).
To run on local copies instead, set `LOCAL_DATA_DIR` (same folder layout as the bucket).

In [1]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (11, 3.5), "axes.spines.top": False, "axes.spines.right": False})

## 0. Config

Edit these if the bucket layout differs. Section 1 prints what's actually there.

In [2]:
PROJECT_ID = os.getenv("GCP_PROJECT_ID", "ba882-team10-inspections")
BUCKET = os.getenv("GCS_LANDING_BUCKET", "ba882-team10-inspections-landing")
RAW_PREFIX = "raw"

# Set to a local folder (e.g. "../../data/raw") to skip GCS
LOCAL_DATA_DIR = os.getenv("LOCAL_DATA_DIR")

# Folder names under raw/ for each source, and how to combine its daily load_date partitions.
# raw_api_ingest loads incrementally: each run re-pulls the last 30 days into a new partition, so one
# partition is NOT the whole table. read_source() reads every partition and keeps the newest copy of
# each record, the same rule as the dbt staging models.
#   key  = column that identifies a record (newest partition wins)
#   None = each partition is a full snapshot, so only the latest one is read (matches stg_food_licenses)
SOURCES = {
    "inspections": {"folder": "food_inspections", "key": "_id"},
    "licenses": {"folder": "food_licenses", "key": None},
    # Legacy feed, not requests_311_new: the new system only carries public-works types (litter, trash,
    # street lights) and has none of the food-safety ones (Rodent Activity, Unsanitary Conditions, Food Alert).
    "requests_311": {"folder": "requests_311_legacy", "key": "case_enquiry_id"},
}

# Result codes that are admin records, not inspections (see section 2)
NON_INSPECTION_RESULTS = ["HE_NotReq", "HE_OutBus", "DATAERR", "HE_Misc", "HE_Hold", "Closed"]

# A visit within this many days after a failed inspection is treated as a follow-up, not routine.
# 90% of re-visits after a fail happen within 17 days; the histogram in section 6 flattens out by ~30.
FOLLOWUP_WINDOW_DAYS = 30

# Boston switched to the 2013 FDA Food Code during 2019 and the "***" level changed meaning (section 2e).
# Inspections from this date on are the "current era": used for the driver analysis and as a model flag.
FDA_ERA_START = "2020-01-01"

# Time-based split: last TEST_MONTHS = test, the VALID_MONTHS before that = validation
TEST_MONTHS = 12
VALID_MONTHS = 12

# 311 features (spatial counts). Turn off if 311 isn't loaded yet.
BUILD_311_FEATURES = True
RADIUS_M = 250
WINDOW_311_DAYS = 90

## 1. Load the data

In [3]:
if LOCAL_DATA_DIR:
    import fsspec

    fs = fsspec.filesystem("file")
    ROOT = os.path.abspath(os.path.join(LOCAL_DATA_DIR))
else:
    import gcsfs

    fs = gcsfs.GCSFileSystem(project=PROJECT_ID)
    ROOT = f"{BUCKET}/{RAW_PREFIX}"

print(f"Reading from: {ROOT}\n")
for src in sorted(fs.ls(ROOT)):
    name = src.rstrip("/").split("/")[-1]
    parts = sorted(p.rstrip("/").split("/")[-1] for p in fs.ls(src) if "load_date=" in p)
    n_files = len(fs.find(src))
    latest = parts[-1] if parts else "(no load_date partitions)"
    print(f"{name:<28} files={n_files:<5} partitions={len(parts):<4} latest={latest}")

Reading from: ba882-team10-inspections-landing/raw



food_inspections             files=275   partitions=5    latest=load_date=2026-10-23
food_licenses                files=5     partitions=5    latest=load_date=2026-10-23


neighborhoods                files=1     partitions=1    latest=load_date=2026-10-13
requests_311_legacy          files=24    partitions=5    latest=load_date=2026-10-23


requests_311_new             files=15    partitions=5    latest=load_date=2026-10-23


In [4]:
DATA_EXT = (".json", ".jsonl", ".ndjson", ".csv", ".parquet")
METADATA_COLS = ["_id", "_load_date", "_loaded_at", "_partition"]


def list_partitions(folder):
    base = f"{ROOT}/{folder}"
    return sorted(p.rstrip("/") for p in fs.ls(base) if "load_date=" in p.rstrip("/").split("/")[-1])


def read_file(path):
    with fs.open(path) as fh:
        if path.endswith(".csv"):
            return pd.read_csv(fh, low_memory=False)
        if path.endswith(".parquet"):
            return pd.read_parquet(fh)
        try:
            return pd.read_json(fh, lines=True, dtype=False)
        except ValueError:
            pass
    with fs.open(path) as fh:  # fall back to a regular JSON array
        return pd.read_json(fh, dtype=False)


def read_partition(path):
    files = [f for f in fs.find(path) if f.endswith(DATA_EXT)]
    if not files:
        return None
    df = pd.concat([read_file(f) for f in files], ignore_index=True)
    df.columns = [str(c).strip().lower() for c in df.columns]
    # Taken from the folder name: older partitions predate the _load_date column
    df["_partition"] = path.split("load_date=")[-1]
    return df


def read_source(key):
    # Load a source across its load_date partitions, keeping the newest copy of each record.
    # Returns None if it isn't in the bucket yet.
    cfg = SOURCES[key]
    folder = cfg["folder"]
    if not fs.exists(f"{ROOT}/{folder}"):
        print(f"[{key}] not found at {ROOT}/{folder}, skipping")
        return None
    parts = list_partitions(folder)
    if not parts:
        print(f"[{key}] no load_date partitions under {ROOT}/{folder}, skipping")
        return None
    if cfg["key"] is None:
        parts = parts[-1:]
    df = pd.concat([d for d in map(read_partition, parts) if d is not None], ignore_index=True)
    n_read = len(df)
    if cfg["key"]:
        record = df[cfg["key"]].astype(str).str.strip()
        df = (
            df.assign(_record=record)
            .sort_values("_partition", kind="stable")
            .drop_duplicates("_record", keep="last")
            .drop(columns="_record")
            .sort_index()
        )
    df = df.replace({"": np.nan}).reset_index(drop=True)
    span = parts[0].split("=")[-1] if len(parts) == 1 else f"{parts[0].split('=')[-1]} to {parts[-1].split('=')[-1]}"
    print(f"[{key}] {len(df):,} rows x {df.shape[1]} cols from {len(parts)} partition(s), {span} "
          f"({n_read - len(df):,} repeat copies dropped)")
    return df


insp_raw = read_source("inspections")
lic_raw = read_source("licenses")
r311_raw = read_source("requests_311")

[inspections] 905,779 rows x 30 cols from 5 partition(s), 2026-10-13 to 2026-10-23 (1,816,011 repeat copies dropped)


[licenses] 3,357 rows x 17 cols from 1 partition(s), 2026-10-23 (0 repeat copies dropped)


[requests_311] 190,414 rows x 33 cols from 5 partition(s), 2026-10-13 to 2026-10-23 (33,550 repeat copies dropped)


### Helpers

Field names on data.boston.gov aren't documented consistently, so `pick()` takes the first name that exists
and fails loudly with the real column list if none do. If you hit a `KeyError`, add the real name to the list.

In [5]:
def pick(df, *candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of {candidates} found. Columns are: {list(df.columns)}")
    return None


def to_naive_datetime(s):
    # format="mixed": 311 open_dt mixes timestamps with and without fractional seconds, and pandas'
    # inferred format drops ~75% of them as NaT
    out = pd.to_datetime(s, errors="coerce", utc=True, format="mixed")
    return out.dt.tz_convert(None)


def parse_lat_lon(df):
    # Return (lat, lon) Series from lat/long columns or a '(lat, lon)' location string.
    lat_c = pick(df, "latitude", "lat", "y", required=False)
    lon_c = pick(df, "longitude", "long", "lon", "lng", "x", required=False)
    if lat_c and lon_c:
        return pd.to_numeric(df[lat_c], errors="coerce"), pd.to_numeric(df[lon_c], errors="coerce")
    loc_c = pick(df, "location", "geocoded_location", required=False)
    if loc_c:
        nums = df[loc_c].astype(str).str.extract(r"(-?\d+\.\d+)[,\s]+(-?\d+\.\d+)").astype(float)
        return nums[0], nums[1]
    return pd.Series(np.nan, index=df.index), pd.Series(np.nan, index=df.index)


def in_boston(lat, lon):
    return lat.between(42.2, 42.42) & lon.between(-71.2, -70.95)


def rate_table(df, by, min_n=100):
    # Critical-violation rate per group, with lift over the overall rate of df
    t = df.groupby(by, observed=True)["target"].agg(inspections="size", critical_rate="mean")
    t["lift"] = t["critical_rate"] / df["target"].mean()
    return t[t["inspections"] >= min_n].round(3)


C = {
    "license": pick(insp_raw, "licenseno", "license_no", "license_number", "licno"),
    "business": pick(insp_raw, "businessname", "business_name", "dbaname", required=False),
    "address": pick(insp_raw, "address", required=False),
    "city": pick(insp_raw, "city", required=False),
    "state": pick(insp_raw, "state", required=False),
    "date": pick(insp_raw, "resultdttm", "result_dttm", "resultdate", "inspection_date", "inspdate"),
    "viol_date": pick(insp_raw, "violdttm", "viol_dttm", required=False),
    "result": pick(insp_raw, "result", "result_code", "inspection_result"),
    "level": pick(insp_raw, "viollevel", "viol_level", "violation_level", "level"),
    "viol_code": pick(insp_raw, "violation", "violation_code", "viol_code", required=False),
    "viol_desc": pick(insp_raw, "violdesc", "violation_description", "viol_desc", required=False),
    "viol_status": pick(insp_raw, "violstatus", "viol_status", "violation_status", required=False),
    "licstatus": pick(insp_raw, "licstatus", "license_status", required=False),
    "licensecat": pick(insp_raw, "licensecat", "license_cat", "license_category", required=False),
    "descript": pick(insp_raw, "descript", "description", "license_description", required=False),
}
C

{'license': 'licenseno',
 'business': 'businessname',
 'address': 'address',
 'city': 'city',
 'state': 'state',
 'date': 'resultdttm',
 'viol_date': 'violdttm',
 'result': 'result',
 'level': 'viol_level',
 'viol_code': 'violation',
 'viol_desc': 'violdesc',
 'viol_status': 'viol_status',
 'licstatus': 'licstatus',
 'licensecat': 'licensecat',
 'descript': 'descript'}

## 2. Data quality: what cleaning does the inspection feed need?

**Question:** the raw feed has one row per *violation*, pulled straight from data.boston.gov. Is it clean enough
to model on, and what has to be fixed in staging (dbt `stg_food_inspections`)?

### 2a. Nulls and cardinality

In [6]:
quality = pd.DataFrame({
    "null_rate": insp_raw.isna().mean(),
    "n_unique": insp_raw.nunique(),
    "example": insp_raw.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None),
}).sort_values("null_rate", ascending=False)
print(f"{len(insp_raw):,} rows, {insp_raw.drop(columns=METADATA_COLS, errors='ignore').duplicated().sum():,} exact duplicate rows")
quality

905,779 rows, 0 exact duplicate rows


,null_rate,n_unique,example
_loaded_at,1.000000,0,NaN
dbaname,0.990611,117,Saab 1 Enterprises
status_date,0.585723,56580,2018-03-23 20:15:45+00
namefirst,0.425157,4076,Kenneth Pasquariello
legalowner,0.415898,2800,KHOSLA VIPAN
property_id,0.174872,4516,156226
comments,0.097727,368493,One staff person without hair restraint. Provide
location,0.088511,4867,"(42.35925954972639, -71.05890048027378)"
viol_level,0.076527,4,*
violdesc,0.076527,454,Clean Cloths Hair Restraint


### 2b. Categorical codes

In [7]:
for key in ["result", "level", "viol_status", "licstatus", "licensecat", "state"]:
    col = C.get(key)
    if col:
        print(f"--- {key} ({col}) ---")
        print(insp_raw[col].value_counts(dropna=False).head(25).to_string(), "\n")

print(f"--- city ({C['city']}): {insp_raw[C['city']].nunique()} distinct values, top 20 ---")
print(insp_raw[C["city"]].value_counts().head(20).to_string())

--- result (result) ---
result
HE_Fail       378334
HE_Pass       288759
HE_Filed       93637
HE_FailExt     77313
HE_Hearing     28012
HE_NotReq      24067
HE_TSOP         7713
HE_VolClos      3128
HE_OutBus       2549
Pass             964
HE_Closure       709
Fail             238
HE_FAILNOR       158
HE_Misc          128
DATAERR           42
HE_Hold           17
Failed             6
Closed             2
PassViol           2
NoViol             1 

--- level (viol_level) ---
viol_level
*       591118
***     128849
**      116494
NaN      69317
1919         1 

--- viol_status (viol_status) ---
viol_status
Fail    462957
Pass    375091
NaN      67731 

--- licstatus (licstatus) ---
licstatus
Inactive    455091
Active      450659
Deleted         29 

--- licensecat (licensecat) ---
licensecat
FS     444317
FT     343797
RF     109207
MFW      8458 

--- state (state) ---
state
MA     891240
Ma      12946
NaN      1185
ma        408 

--- city (city): 36 distinct values, top 20 ---
city


**Reading the result codes.** Several codes aren't inspections at all. `HE_NotReq` (re-inspection not required)
has a critical violation in 0.3% of cases, so it's an admin record. `HE_OutBus` means the place closed. These and a
handful of rare codes (`DATAERR`, `HE_Misc`, `HE_Hold`, `Closed`) are in `NON_INSPECTION_RESULTS` and get dropped.
`HE_Hearing`, `HE_TSOP` (temporary suspension), `HE_Closure` and `HE_VolClos` are enforcement outcomes that follow a
bad inspection, so they stay as history.

### 2c. Dates

In [8]:
date_cols = [c for c in insp_raw.columns if c.endswith("dttm") or c.endswith("_date")]
parsed = {c: to_naive_datetime(insp_raw[c]) for c in date_cols}
display(pd.DataFrame({
    c: {"non_null": insp_raw[c].notna().mean(), "parsed": s.notna().mean(), "min": s.min(), "max": s.max()}
    for c, s in parsed.items()
}).T)

rt, vt = parsed[C["date"]], parsed.get(C["viol_date"])
if vt is not None:
    carried = vt.notna() & (vt.dt.normalize() < rt.dt.normalize())
    print(f"Violation dated before its inspection (carried over from an earlier visit): {carried.mean():.2%} of rows")

,non_null,parsed,min,max
issdttm,0.999074,0.999074,2007-01-01 20:15:05,2026-10-19 19:08:48
expdttm,0.999312,0.999312,2007-01-11 17:23:25.990000,2027-01-01 04:59:00
resultdttm,0.992936,0.992936,2006-04-04 12:49:18,2026-10-19 18:55:13
violdttm,0.931557,0.931557,2006-11-21 05:00:00,2026-10-19 18:55:13
status_date,0.414277,0.414277,2006-12-27 05:00:00,2026-10-19 19:56:01
_load_date,1.0,1.0,2026-10-21 00:00:00,2026-10-23 00:00:00


Violation dated before its inspection (carried over from an earlier visit): 0.32% of rows


### 2d. Coordinates

In [9]:
lat, lon = parse_lat_lon(insp_raw)
ok = in_boston(lat, lon)
print(f"Rows with a parseable location: {lat.notna().mean():.1%}")
print(f"Rows inside the Boston bounding box: {ok.mean():.1%}")
has_coords = ok.groupby(insp_raw[C["license"]]).any()
print(f"Establishments with no usable coordinates at all: {1 - has_coords.mean():.1%}")
print("\nMost common points (if one dwarfs the rest it's a default geocode; these look like multi-tenant sites):")
print(pd.DataFrame({"lat": lat[ok], "lon": lon[ok]}).value_counts().head(3).to_string())

Rows with a parseable location: 91.1%
Rows inside the Boston bounding box: 91.1%
Establishments with no usable coordinates at all: 13.0%

Most common points (if one dwarfs the rest it's a default geocode; these look like multi-tenant sites):
lat        lon       
42.347297  -71.081457    3266
42.355831  -71.060401    3112
42.352410  -71.125409    3041


### 2e. The violation code system changed in 2019

Violation codes come in two formats. The newer one (e.g. `590.003/3-501.16-P`) is the 2013 FDA Food Code, whose
suffix is the FDA severity: **P** = Priority, **PF** = Priority Foundation, **C** = Core. `M-` codes are
person-in-charge / management items.

In [10]:
code_str = insp_raw[C["viol_code"]].astype(str)
fda = code_str.str.match(r"^\d{3}\.\d{3}/|^M-")
era = pd.Series(np.where(fda, "FDA Food Code", np.where(insp_raw[C["viol_code"]].isna(), "no violation", "old Boston code")), index=insp_raw.index)
year = rt.dt.year

suffix = code_str.str.extract(r"-(P|PF|C)$")[0].where(~code_str.str.startswith("M-"), "M (management)")
print("FDA severity suffix vs Boston star level (FDA-format rows only):")
display(pd.crosstab(suffix[fda].rename("FDA suffix"), insp_raw.loc[fda, C["level"]]))

share = pd.crosstab(year, era, normalize="index")
level_share = pd.crosstab(year, insp_raw[C["level"]].where(insp_raw[C["level"]].isin(["*", "**", "***"])), normalize="index")
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
share.drop(columns="no violation", errors="ignore").plot(ax=axes[0], title="Share of violation rows by code system")
level_share.plot(ax=axes[1], title="Share of violation rows by star level")
for ax in axes:
    ax.set_xlabel("")
plt.show()

FDA severity suffix vs Boston star level (FDA-format rows only):


viol_level,*,**,***
FDA suffix,,,
C,171781,0,0
M (management),0,135,26075
P,0,0,26826
PF,0,71936,0


### Answer: cleaning needed

The feed is usable but needs these fixes in staging. None of them are blockers.

| Issue | Scale | Fix |
|---|---|---|
| Grain is one row per violation, not per inspection | ~4 rows per inspection | Aggregate to `license` + inspection date (section 4) |
| Non-inspection result codes (`HE_NotReq`, `HE_OutBus`, `DATAERR`, …) | ~10% of inspections | Drop; list in `NON_INSPECTION_RESULTS` |
| **`***` changed meaning in 2019**: old "critical" became FDA Priority (P) + management (M); old-critical items not rated P became Priority Foundation (`**`) | `***` share of rows fell from ~18% to ~11% | Compare rates within one era; give the model an `fda_code_era` flag (section 11 tests the options) |
| Timestamps in mixed formats (with/without fractional seconds, `+00` offsets) | All date columns | Parse with `format="mixed"` / `SAFE.PARSE_TIMESTAMP` with fallbacks |
| `resultdttm` missing | 0.7% of rows | Drop those rows |
| `viol_level`, `violation`, `violdesc` null | ~7% of rows | These are inspections with no violation, not missing data. Count as 0 violations |
| Junk star level (`1919`) | 1 row | Treat as unknown |
| `state` casing (`MA` / `Ma` / `ma`) | 1.5% of rows | Upper-case |
| `city` is a neighborhood name, 45% just "BOSTON", with variants like `MISSION HILL/` | 36 values | Upper-case, trim, strip trailing `/`; later replace with a spatial join to the `neighborhoods` GeoJSON |
| `location` missing | ~9% of rows, ~13% of establishments | Leave as missing; spatial features will be NaN for these |
| `dbaname` (99% null), `status_date` (59% null) | — | Not needed; drop |
| `licstatus`, `issdttm`, `expdttm` are **current** snapshot values repeated on every historical row | All rows | Never use as features (leak the future); see section 9 |
| Violations carried over from an earlier visit | 0.3% of rows | Ignore |

## 3. Hypothesis: the inspections feed is still being updated

The "legacy" name on data.boston.gov raised the question of whether the dataset is frozen. If it is, the project
needs a different source.

In [11]:
insp = insp_raw.copy()
insp["insp_dt"] = to_naive_datetime(insp[C["date"]])

latest = insp["insp_dt"].max()
age_days = (pd.Timestamp.now() - latest).days
print(f"Date range: {insp['insp_dt'].min():%Y-%m-%d} to {latest:%Y-%m-%d}")
print(f"Latest record is {age_days} days old")
print(f"Unparseable dates: {insp['insp_dt'].isna().sum():,}")
if age_days > 60:
    print("\n*** WARNING: no inspections in the last 60 days. Tell the team before building on this. ***")

monthly = insp.dropna(subset=["insp_dt"]).set_index("insp_dt").resample("MS").size()
ax = monthly[monthly.index >= latest - pd.DateOffset(years=5)].plot(title="Violation rows per month (last 5 years)")
ax.set_xlabel("")
plt.show()

Date range: 2006-04-04 to 2026-10-19
Latest record is 3 days old
Unparseable dates: 6,398


**Answer: supported.** The latest inspection is only days older than the snapshot, and monthly volume over the
last five years is steady. The feed is live, so it can drive both training and weekly scoring.
**Record the latest date and row counts in `docs/data_sources.md`.**

## 4. Reshape: violation rows → one row per inspection

Grain = `license` + inspection **date**. An inspection with five violations shows up five times in the raw data,
and a clean inspection usually shows up once with an empty violation. Cleaning from section 2 is applied here.

In [12]:
v = insp.dropna(subset=["insp_dt"]).copy()
n_before = len(v)
v = v[~v[C["result"]].isin(NON_INSPECTION_RESULTS)]
print(f"Dropped {n_before - len(v):,} rows with non-inspection result codes")

v["license"] = v[C["license"]].astype(str).str.strip()
v = v[v["license"].ne("") & v["license"].str.lower().ne("nan")]
v["insp_date"] = v["insp_dt"].dt.normalize()

level_str = v[C["level"]].astype(str).str.strip()
v["is_critical"] = level_str.eq("***")
v["is_major"] = level_str.eq("**")
v["has_violation"] = level_str.isin(["*", "**", "***"])
v["result_fail"] = v[C["result"]].astype(str).str.contains("fail", case=False)
v["city_clean"] = v[C["city"]].astype(str).str.upper().str.strip().str.rstrip("/").str.strip()
v["lat"], v["lon"] = parse_lat_lon(v)

agg = {
    "n_rows": ("license", "size"),
    "n_violations": ("has_violation", "sum"),
    "n_critical": ("is_critical", "sum"),
    "n_major": ("is_major", "sum"),
    "result_fail": ("result_fail", "max"),
    "result": (C["result"], "first"),
    "city": ("city_clean", "first"),
    "lat": ("lat", "median"),
    "lon": ("lon", "median"),
}
if C["business"]:
    agg["business"] = (C["business"], "first")
if C["licensecat"]:
    agg["licensecat"] = (C["licensecat"], "first")

inspections = (
    v.groupby(["license", "insp_date"], as_index=False)
    .agg(**agg)
    .sort_values(["license", "insp_date"])
    .reset_index(drop=True)
)
mixed = v.groupby(["license", "insp_date"])[C["result"]].nunique().gt(1).mean()
print(f"{len(v):,} violation rows -> {len(inspections):,} inspections across {inspections['license'].nunique():,} establishments")
print(f"Inspections whose rows disagree on the result code: {mixed:.2%} (taking the first)")
inspections.head()

Dropped 24,641 rows with non-inspection result codes


874,740 violation rows -> 195,653 inspections across 10,729 establishments
Inspections whose rows disagree on the result code: 0.29% (taking the first)


,license,insp_date,n_rows,n_violations,n_critical,n_major,result_fail,result,city,lat,lon,business,licensecat
0,105070,2013-08-22,1,0,0,0,False,HE_Pass,BOSTON,NaN,NaN,Lady C & J Soulfood and Catering,MFW
1,105409,2013-08-26,3,3,2,0,True,HE_Fail,JAMAICA PLAIN,42.32199,-71.10926,The Goods JP,FT
2,105409,2013-09-03,3,3,2,0,False,HE_Pass,JAMAICA PLAIN,42.32199,-71.10926,The Goods JP,FT
3,105409,2014-03-04,3,3,0,0,True,HE_Fail,JAMAICA PLAIN,42.32199,-71.10926,The Goods JP,FT
4,105409,2014-03-11,3,3,0,0,False,HE_Pass,JAMAICA PLAIN,42.32199,-71.10926,The Goods JP,FT


## 5. Distributions

In [13]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7))

per_year = inspections.groupby(inspections["insp_date"].dt.year).agg(inspections=("license", "size"), establishments=("license", "nunique"))
per_year.plot(ax=axes[0, 0], marker="o", title="Inspections and establishments per year")
axes[0, 0].set_xlabel("")

per_lic = inspections.groupby("license").size()
per_lic.clip(upper=100).hist(bins=50, ax=axes[0, 1])
axes[0, 1].set_title(f"Inspections per establishment (median {per_lic.median():.0f}, clipped at 100)")

inspections["n_violations"].clip(upper=25).hist(bins=26, ax=axes[1, 0])
axes[1, 0].set_title(f"Violations per inspection (median {inspections['n_violations'].median():.0f}, clipped at 25)")

inspections["n_critical"].clip(upper=10).value_counts().sort_index().plot.bar(ax=axes[1, 1], rot=0)
axes[1, 1].set_title("Critical (***) violations per inspection (10 = 10+)")
plt.tight_layout()
plt.show()

print("Violations per inspection:", inspections["n_violations"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).round(1).to_dict())
print(f"Inspections with zero violations: {inspections['n_violations'].eq(0).mean():.1%}")
print(f"Inspections per establishment: median {per_lic.median():.0f}, establishments with only 1: {per_lic.eq(1).mean():.1%}")
print("Per year:", per_year["inspections"].to_dict())

Violations per inspection: {'count': 195653.0, 'mean': 4.2, 'std': 4.4, 'min': 0.0, '25%': 1.0, '50%': 3.0, '75%': 6.0, '90%': 10.0, '99%': 19.0, 'max': 66.0}
Inspections with zero violations: 18.8%
Inspections per establishment: median 12, establishments with only 1: 6.8%
Per year: {2006: 14, 2007: 4472, 2008: 8736, 2009: 9222, 2010: 9058, 2011: 9143, 2012: 10034, 2013: 9016, 2014: 9035, 2015: 9874, 2016: 10985, 2017: 9917, 2018: 10680, 2019: 9992, 2020: 7458, 2021: 11047, 2022: 12162, 2023: 10719, 2024: 11387, 2025: 12271, 2026: 10431}


In [14]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
inspections["result"].value_counts().head(10).sort_values().plot.barh(ax=axes[0], title="Result code (per inspection)")
inspections["licensecat"].value_counts().sort_values().plot.barh(ax=axes[1], title="License category")
inspections["city"].value_counts().head(15).sort_values().plot.barh(ax=axes[2], title="Neighborhood (city field)")
plt.tight_layout()
plt.show()
print("License category share:", inspections["licensecat"].value_counts(normalize=True).round(3).to_dict())
print("License categories: FS = Eating & Drinking, FT = Eating & Drinking w/ Take Out, RF = Retail Food, MFW = Mobile Food Walk On")

License category share: {'FS': 0.441, 'FT': 0.383, 'RF': 0.161, 'MFW': 0.015}
License categories: FS = Eating & Drinking, FT = Eating & Drinking w/ Take Out, RF = Retail Food, MFW = Mobile Food Walk On


In [15]:
crit_rows = v[v["is_critical"]]
print("Most common critical (***) violations:")
crit_rows[C["viol_desc"]].value_counts().head(15).to_frame("rows")

Most common critical (***) violations:


,rows
violdesc,
Cold Holding,13554
PIC Performing Duties,11333
Food Contact Surfaces Clean,9176
Location Accessible,8162
(A)(2) and (B) Time/Temperature Control for Safety Food Hot and Cold Holding (P),7614
PIC Knowledge,7376
Separation Segregation Cross Contamination,6682
Hot Holding,6196
Packaged and Unpackaged Food-Separation Packaging and Segregation (P),3812


**What the distributions show**

- About 9–12k inspections a year (routine plus follow-ups) across 3–4k establishments. 2020 dips to 7.5k (COVID).
- Establishments have a long history (median 12 inspections; only 7% have a single one), so there's plenty of
  per-establishment history for features. Brand-new places are the cold-start case.
- Violations per inspection are right-skewed: median 3, 90th percentile 10, 99th percentile 19. About 1 in 5
  inspections (19%) records none.
- Restaurants (`FS` 44%, `FT` 38%) are 82% of inspections. Retail (`RF`, 16%) and mobile (`MFW`, 2%) make up the
  rest.
- The most common critical violations are about temperature control (cold/hot holding), the person in charge, and
  food-contact surfaces.

## 6. Target definition

**Hypothesis:** "the routine inspection records at least one `***` violation" is a usable binary target. It
should be common enough to learn, stable enough to train on, and match what an inspector would want flagged.

- **`target`** = 1 if the inspection recorded any `***` violation (FDA Priority or management item since 2019)
- **`result_fail`** = the city's own result code contains "fail" (kept to compare against)
- **Follow-ups** = a visit within `FOLLOWUP_WINDOW_DAYS` after a failed one. They're scheduled *because* of a known
  problem, so they're easy to predict and would inflate the metrics. They're excluded from training and evaluation
  but still count as history for features.

The dataset doesn't label inspection type, so the follow-up rule is a heuristic. The histogram below is the
evidence for the window.

In [16]:
g = inspections.groupby("license")
inspections["target"] = (inspections["n_critical"] > 0).astype(int)
inspections["prev_date"] = g["insp_date"].shift(1)
inspections["days_since_prev"] = (inspections["insp_date"] - inspections["prev_date"]).dt.days
inspections["prev_result_fail"] = g["result_fail"].shift(1)
inspections["prev_target"] = g["target"].shift(1)

prev_failed = inspections["prev_result_fail"].eq(True) | inspections["prev_target"].eq(1)
inspections["is_followup"] = prev_failed & inspections["days_since_prev"].le(FOLLOWUP_WINDOW_DAYS)
inspections["is_routine"] = ~inspections["is_followup"]

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
inspections.loc[prev_failed, "days_since_prev"].clip(upper=120).hist(bins=60, ax=axes[0])
axes[0].axvline(FOLLOWUP_WINDOW_DAYS, color="red", ls="--")
axes[0].set_title("Days to next visit after a failed inspection (red = cutoff)")
inspections.loc[~prev_failed & inspections["prev_date"].notna(), "days_since_prev"].clip(upper=730).hist(bins=73, ax=axes[1])
axes[1].set_title("Days to next visit after a passed inspection")
plt.show()

after_fail = inspections.loc[inspections["prev_result_fail"].eq(True), "days_since_prev"]
print("Days to next visit after a result-code fail:", after_fail.describe(percentiles=[0.5, 0.75, 0.9]).round(0).to_dict())
print(f"Share of next visits within {FOLLOWUP_WINDOW_DAYS} days of a fail: {after_fail.le(FOLLOWUP_WINDOW_DAYS).mean():.1%}")
print(inspections["is_followup"].value_counts(normalize=True).rename("share").to_string())
print("\nCritical rate, follow-up vs routine:", inspections.groupby("is_followup")["target"].mean().round(3).to_dict())

Days to next visit after a result-code fail: {'count': 74628.0, 'mean': 15.0, 'std': 39.0, 'min': 1.0, '50%': 8.0, '75%': 13.0, '90%': 17.0, 'max': 2736.0}
Share of next visits within 30 days of a fail: 95.6%
is_followup
False    0.617225
True     0.382775

Critical rate, follow-up vs routine: {False: 0.291, True: 0.494}


In [17]:
routine = inspections[inspections["is_routine"]].copy()
routine["year"] = routine["insp_date"].dt.year

print("Target vs city result code (routine inspections):")
display(pd.crosstab(routine["target"], routine["result_fail"], margins=True))
print("Critical rate by result code (routine inspections):")
display(rate_table(routine, "result").sort_values("inspections", ascending=False))

Target vs city result code (routine inspections):


result_fail,False,True,All
target,,,
0,54318,31337,85655
1,1940,33167,35107
All,56258,64504,120762


Critical rate by result code (routine inspections):


,inspections,critical_rate,lift
result,,,
HE_Fail,63819,0.516,1.777
HE_Pass,37972,0.020,0.070
HE_Filed,16715,0.019,0.065
HE_FailExt,650,0.286,0.984
HE_TSOP,607,0.707,2.431
HE_VolClos,409,0.509,1.749
HE_Hearing,294,0.435,1.498
Pass,159,0.057,0.195
HE_Closure,101,0.792,2.725


In [18]:
by_year = routine.groupby("year")["target"].agg(inspections="size", critical_rate="mean")
ax = by_year["critical_rate"].plot(marker="o", title="Critical-violation rate per routine inspection, by year")
ax.axvspan(2018.5, 2019.5, color="orange", alpha=0.2, label="FDA Food Code cutover")
ax.axvspan(2019.6, 2020.9, color="grey", alpha=0.2, label="COVID")
ax.set_ylim(0, None)
ax.set_xlabel("")
ax.legend()
plt.show()

recent = routine[routine["insp_date"] >= FDA_ERA_START]
print(f"Base rate, all years: {routine['target'].mean():.1%} of {len(routine):,} routine inspections")
print(f"Base rate since {FDA_ERA_START}: {recent['target'].mean():.1%} of {len(recent):,} routine inspections")
by_year.round(3).T

Base rate, all years: 29.1% of 120,762 routine inspections
Base rate since 2020-01-01: 21.7% of 45,866 routine inspections


year,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
inspections,13.000,2800.000,5268.000,5964.000,5886.000,6066.000,6332.000,5728.000,5474.000,5919.000,6405.000,6308.000,6500.000,6233.000,4744.000,6982.000,7322.000,6608.000,6756.000,7403.000,6051.000
critical_rate,0.231,0.381,0.376,0.287,0.283,0.295,0.333,0.307,0.351,0.363,0.381,0.328,0.401,0.302,0.204,0.208,0.223,0.199,0.235,0.202,0.246


### Answer: target

**Supported. "Any `***` violation at a routine inspection" works as the target, but the 2019 code change has to be handled.**

- **Follow-ups are easy to separate.** After a result-code fail, the next visit comes a median of 8 days later,
  90% within 17 days and 96% within 30. After a pass it's months. With a 30-day window, follow-ups are 38% of all
  inspections, with a 49% critical rate vs 29% for routine ones. Excluding them avoids an easy, inflated win.
- **The base rate is learnable.** 21.7% of 45.7k routine inspections since 2020 found a critical violation, about
  1 in 5. That's common enough to learn and rare enough that ranking matters.
- **The target is a stricter version of the city's verdict.** ~95% of routine inspections with a `***` were graded
  as fails, but about half of all fails had no `***`; they're driven by volumes of `*`/`**` items. `***` captures
  *severity*, which is what triage needs.
- **The rate isn't stable over time.** It ran 28–40% through 2019, then 20–25% from 2020. Most of the drop lines
  up with the 2019 FDA Food Code cutover (section 2e), when part of the old "critical" category became `**`.
  COVID adds noise in 2020. So driver rates (section 7) are measured within the current era, and the model gets
  an `fda_code_era` flag. Section 11 shows this keeps predictions calibrated without throwing away history.
- Alternative target to keep in mind: any `**` or `***` (Priority + Priority Foundation). It's more common and
  closer to the old "critical" definition. Worth a robustness check later.

## 7. Hypotheses: what predicts a critical violation?

Each hypothesis below is a candidate feature. They're tested on routine inspections since `FDA_ERA_START`, so they
reflect the current target definition. **Lift** = group rate ÷ overall rate (1.0 = no signal).

In [19]:
rg = routine.groupby("license")
routine["prev_routine_target"] = rg["target"].shift(1)
routine["prev2_routine_target"] = rg["target"].shift(2)
routine["n_prior_routine"] = rg.cumcount()
routine["days_since_prev_routine"] = (routine["insp_date"] - rg["insp_date"].shift(1)).dt.days
recent = routine[routine["insp_date"] >= FDA_ERA_START].copy()
base = recent["target"].mean()
print(f"Base rate: {base:.1%} over {len(recent):,} routine inspections")

Base rate: 21.7% over 45,866 routine inspections


**H1: Repeat offenders.** Places that had a critical violation last time are more likely to have one again.

In [20]:
display(rate_table(recent, "prev_routine_target"))
display(rate_table(recent, ["prev_routine_target", "prev2_routine_target"]))

,inspections,critical_rate,lift
prev_routine_target,,,
0.0,33706,0.168,0.774
1.0,9566,0.420,1.937


inspections  critical_rate   lift
prev_routine_target prev2_routine_target                                   
0.0                 0.0                         26215          0.143  0.661
                    1.0                          5391          0.276  1.275
1.0                 0.0                          5253          0.367  1.694
                    1.0                          4055          0.484  2.234

**H2: License category.** Full-service restaurants handle more raw food and hot/cold holding than retail or mobile vendors.

In [21]:
rate_table(recent, "licensecat")

,inspections,critical_rate,lift
licensecat,,,
FS,20365,0.259,1.193
FT,18016,0.202,0.932
MFW,989,0.080,0.368
RF,6496,0.147,0.680


**H3: Neighborhood.** Risk varies across the city.

In [22]:
t = rate_table(recent, "city", min_n=300).sort_values("critical_rate")
display(t.T)
ax = t["critical_rate"].plot.barh(figsize=(8, 5), title="Critical rate by neighborhood (≥300 inspections)")
ax.axvline(base, color="red", ls="--")
plt.show()

city,CHARLESTOWN,EAST BOSTON,MATTAPAN,HYDE PARK,BOSTON,BRIGHTON,ROSLINDALE,WEST ROXBURY,ROXBURY,SOUTH BOSTON,MISSION HILL,ALLSTON,DORCHESTER,JAMAICA PLAIN
inspections,869.000,3256.000,716.000,807.000,21890.000,2016.000,1150.000,973.000,3135.000,1577.000,753.000,1893.000,5159.000,1558.000
critical_rate,0.114,0.168,0.177,0.178,0.195,0.213,0.216,0.222,0.236,0.259,0.266,0.267,0.296,0.297
lift,0.525,0.775,0.818,0.823,0.901,0.984,0.995,1.024,1.090,1.196,1.225,1.233,1.367,1.371


**H4: Seasonality.** Summer heat makes cold holding harder.

In [23]:
by_month = rate_table(recent, recent["insp_date"].dt.month.rename("month"))
by_dow = rate_table(recent, recent["insp_date"].dt.dayofweek.rename("weekday (0=Mon)"), min_n=50)
display(by_month.T, by_dow.T)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
by_month["critical_rate"].plot.bar(ax=axes[0], rot=0, title="By month")
by_dow["critical_rate"].plot.bar(ax=axes[1], rot=0, title="By weekday")
for ax in axes:
    ax.axhline(base, color="red", ls="--")
plt.show()

month,1,2,3,4,5,6,7,8,9,10,11,12
inspections,4384.000,3719.000,4418.000,3792.000,3434.000,3472.000,3792.000,4146.000,3758.000,4257.000,3623.000,3071.000
critical_rate,0.226,0.237,0.206,0.193,0.222,0.221,0.239,0.237,0.203,0.226,0.194,0.189
lift,1.044,1.095,0.949,0.892,1.025,1.021,1.103,1.093,0.934,1.040,0.896,0.873


weekday (0=Mon),0,1,2,3,4,5,6
inspections,8346.000,9573.000,9896.000,9757.000,7745.000,450.000,99.000
critical_rate,0.216,0.223,0.225,0.222,0.200,0.082,0.283
lift,0.996,1.027,1.038,1.025,0.924,0.379,1.304


**H5: Time since the last routine inspection.** Longer gaps mean more time for standards to slip.

In [24]:
rate_table(recent, pd.cut(recent["days_since_prev_routine"], [0, 90, 180, 365, 730, 10_000]))

,inspections,critical_rate,lift
days_since_prev_routine,,,
"(0, 90]",6710,0.201,0.929
"(90, 180]",16650,0.238,1.096
"(180, 365]",15703,0.229,1.057
"(365, 730]",3795,0.190,0.875
"(730, 10000]",414,0.109,0.501


**H6: Tenure.** New establishments make more mistakes, or alternatively, establishments with a long track record have settled in.

In [25]:
rate_table(recent, pd.cut(recent["n_prior_routine"], [-1, 0, 2, 5, 10, 20, 500]))

,inspections,critical_rate,lift
n_prior_routine,,,
"(-1, 0]",2594,0.104,0.480
"(0, 2]",4577,0.218,1.004
"(2, 5]",5776,0.227,1.048
"(5, 10]",7627,0.226,1.041
"(10, 20]",11391,0.207,0.953
"(20, 500]",13901,0.237,1.092


### Answer: drivers

| Hypothesis | Verdict | Evidence (routine inspections since 2020; base rate 21.7%) |
|---|---|---|
| H1 repeat offenders | **Strongly supported.** The best single signal | 42.0% after a critical last time vs 16.8% after a clean one (2.5×). Two criticals in a row: 48.5% vs 14.3% after two clean |
| H2 license category | **Supported** | Full-service `FS` 25.9%, take-out `FT` 20.2%, retail `RF` 14.7%, mobile `MFW` 8.1% |
| H3 neighborhood | **Supported in raw rates, little extra value** | From 11% (Charlestown) to 30% (Dorchester, Jamaica Plain). But `city` has near-zero model importance (section 11), so neighborhood gaps are mostly explained by the establishments' own histories |
| H4 seasonality | **Rejected** | Monthly rates stay between 19% and 24% with no summer peak. Saturday visits are rare (448) and low (8%), probably special-purpose visits |
| H5 time since last routine | **Weak, non-monotonic** | 20% under 90 days, ~23% at 3–12 months, 11% after 2+ years. Long gaps are mostly low-risk places the city deprioritises, so this partly encodes inspectors' own judgement |
| H6 new establishments are riskier | **Rejected; the reverse holds** | First-ever inspection: 10.4%, half the base rate; afterwards flat at 21–24%. First inspections are likely pre-opening permit checks of a kitchen that isn't operating yet |

The pattern is that **an establishment's own history dominates**, consistent with the Chicago model this project
builds on. That's why the baseline features below are mostly history features.

## 8. Leakage-safe features

Every feature uses only inspections **before** the current one (`shift(1)` or `closed="left"` windows).
History is computed on *all* inspections, including follow-ups and pre-2020 ones, then we keep the routine rows.

In [26]:
f = inspections.copy()
g = f.groupby("license")

f["n_prior_insp"] = g.cumcount()
f["prior_critical_total"] = g["n_critical"].cumsum() - f["n_critical"]
f["prior_violations_total"] = g["n_violations"].cumsum() - f["n_violations"]
f["prior_critical_rate"] = (g["target"].cumsum() - f["target"]) / f["n_prior_insp"].replace(0, np.nan)
f["prev_n_critical"] = g["n_critical"].shift(1)
f["prev_n_violations"] = g["n_violations"].shift(1)
f["prev_result_fail"] = f["prev_result_fail"].astype(float)
f["days_since_first_insp"] = (f["insp_date"] - g["insp_date"].transform("min")).dt.days
f["month"] = f["insp_date"].dt.month
f["fda_code_era"] = (f["insp_date"] >= FDA_ERA_START).astype(float)

# The visit right before a routine inspection is often a follow-up that passed, which hides the
# original failure. So also look back to the last *routine* inspection.
routine_crit = f["n_critical"].where(f["is_routine"])
routine_date = f["insp_date"].where(f["is_routine"])
f["last_routine_n_critical"] = routine_crit.groupby(f["license"]).shift(1).groupby(f["license"]).ffill()
last_routine_date = routine_date.groupby(f["license"]).shift(1).groupby(f["license"]).ffill()
f["days_since_last_routine"] = (f["insp_date"] - last_routine_date).dt.days
f["had_followup_since_last_routine"] = f["prev_date"].gt(last_routine_date).astype(float).where(last_routine_date.notna())


def rolling_prior(df, col, window):
    r = (
        df.set_index("insp_date")
        .groupby("license")[col]
        .rolling(window, closed="left")
        .sum()
        .rename(f"{col}_{window}")
        .reset_index()
    )
    return df.merge(r, on=["license", "insp_date"], how="left")


for col in ["n_critical", "n_violations"]:
    f = rolling_prior(f, col, "365D")
f = f.rename(columns={"n_critical_365D": "critical_past_365d", "n_violations_365D": "violations_past_365d"})

# Sanity check: features must not peek at the current inspection
assert (f.loc[f["n_prior_insp"] == 0, "prior_critical_total"] == 0).all()
assert f["critical_past_365d"].fillna(0).le(f["prior_critical_total"]).all()
f.filter(like="prior").describe().T

,count,mean,std,min,25%,50%,75%,max
n_prior_insp,195653.0,17.930729,17.539095,0.0,5.000000,13.000000,26.000000,131.0
prior_critical_total,195653.0,14.988592,22.105068,0.0,1.000000,6.000000,20.000000,197.0
prior_violations_total,195653.0,91.969926,123.678055,0.0,11.000000,46.000000,123.000000,1157.0
prior_critical_rate,184924.0,0.387135,0.285146,0.0,0.146341,0.391304,0.592593,1.0


## 9. Licenses: can we join them, and what do they add?

**Hypothesis:** the licenses feed adds establishment attributes we don't already have (age, status), joined on license number.

In [27]:
lic = None
if lic_raw is not None:
    print("License feed columns:", list(lic_raw.columns))
    for col in ["licstatus", "licensecat"]:
        if col in lic_raw:
            print(f"{col}: {lic_raw[col].value_counts().to_dict()}")

    # No license number in this feed, so match on normalized business name + category + street number/name
    def name_key(s):
        return s.astype(str).str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)

    def addr_key(s):
        return s.astype(str).str.upper().str.split().str[:2].str.join(" ")

    def join_key(df):
        return name_key(df[pick(df, "businessname")]) + "|" + df[pick(df, "licensecat")].astype(str) + "|" + addr_key(df[pick(df, "address")])

    per_license = insp_raw.drop_duplicates(C["license"])
    keys = pd.DataFrame({"key": join_key(per_license), "license": per_license[C["license"]].astype(str).str.strip()})
    keys = keys.drop_duplicates("key", keep=False)

    lic = lic_raw.assign(key=join_key(lic_raw)).merge(keys, on="key", how="left")
    print(f"\nLicense rows matched to an inspected license number: {lic['license'].notna().mean():.1%} of {len(lic):,}")

    active_now = per_license[per_license[C["licstatus"]].eq("Active") & per_license[C["licensecat"]].isin(["FS", "FT"])]
    print(f"Active FS/FT licenses in the inspections feed: {len(active_now):,}  (license feed rows: {len(lic_raw):,})")

    issued_c = pick(lic_raw, "license_add_dt_tm", "issdttm", "issue_date", required=False)
    if issued_c:
        lic["license_issued"] = to_naive_datetime(lic[issued_c])
        f = f.merge(lic.dropna(subset=["license"])[["license", "license_issued"]].drop_duplicates("license"), on="license", how="left")
        f["license_age_days"] = (f["insp_date"] - f["license_issued"]).dt.days
        f.loc[f["license_age_days"] < 0, "license_age_days"] = np.nan

print("\nWhy current-status fields leak: critical rate of routine inspections by the license's status *today*")
display(rate_table(routine.merge(insp_raw.drop_duplicates(C["license"])[[C["license"], C["licstatus"]]].astype(str).rename(columns={C["license"]: "license"}), on="license"), C["licstatus"], min_n=50))

License feed columns: ['_id', 'businessname', 'dbaname', 'address', 'city', 'state', 'zip', 'licstatus', 'licensecat', 'descript', 'license_add_dt_tm', 'dayphn_cleaned', 'property_id', 'latitude', 'longitude', '_load_date', '_partition']
licstatus: {'Active': 3357}
licensecat: {'FS': 1762, 'FT': 1595}

License rows matched to an inspected license number: 87.3% of 3,357
Active FS/FT licenses in the inspections feed: 3,357  (license feed rows: 3,357)

Why current-status fields leak: critical rate of routine inspections by the license's status *today*


,inspections,critical_rate,lift
licstatus,,,
Active,63895,0.265,0.910
Inactive,56863,0.320,1.101


### Answer: licenses

**Partly rejected.** The feed has **no license number**, only name, address, category, issue date, phone and
coordinates. It is also just a snapshot of the ~3.4k *currently active* restaurant licenses (`FS`/`FT`), which
matches the inspections feed's own active FS/FT count exactly.

- A composite key (normalized name + category + street number/name) matches ~87% of license rows to a license
  number (after dropping keys that are ambiguous on the inspections side). That's good
  enough for scoring, but the join belongs in dbt with a manual review of the misses.
- The feed adds almost nothing for *training*. Its only new attribute is the original issue date
  (`license_add_dt_tm`), and that only exists for licenses that are still active today. A model using it would learn
  "has an issue date = survived to 2026", which is information from the future. Places that are inactive today had a 32% critical rate vs 27% for still-active ones, so
  survival is correlated with the target, which confirms the leak. **`license_age_days` is computed but kept out of the
  model**, and so are `licstatus`, `issdttm` and `expdttm` from the inspections feed.
- Where it *is* useful: as the **scoring population**. The establishments to rank each week are the active licenses.

## 10. Hypothesis: nearby food-safety 311 complaints raise the risk

Rodent, pest and unsanitary-conditions complaints near a restaurant in the weeks before an inspection may signal problems.

In [28]:
r311 = None
FOOD_311 = r"rodent|rat\b|pest|unsanitary|food|restaurant|dumpster|odor"

if r311_raw is not None:
    R = {
        "type": pick(r311_raw, "type", "case_type", "request_type", "service_name"),
        "reason": pick(r311_raw, "reason", "category", "subject", required=False),
        "opened": pick(r311_raw, "open_dt", "opened", "open_date", "requested_datetime", "created_at"),
    }
    print(R)
    r311 = r311_raw.copy()
    r311["opened"] = to_naive_datetime(r311[R["opened"]])
    r311["lat"], r311["lon"] = parse_lat_lon(r311)
    print(f"Date range: {r311['opened'].min():%Y-%m-%d} to {r311['opened'].max():%Y-%m-%d}  (dates parsed: {r311['opened'].notna().mean():.1%})")

    types = r311[R["type"]].value_counts()
    print("\nFood-safety-related types (edit FOOD_311 to tune):")
    print(types[types.index.str.contains(FOOD_311, case=False, regex=True)].head(30).to_string())

    # Geocoding sanity: a single very common point is usually a default city-hall geocode
    good = in_boston(r311["lat"], r311["lon"])
    top_pt = r311.loc[good].groupby(["lat", "lon"]).size().sort_values(ascending=False).head(3)
    print(f"\nRows with usable Boston coordinates: {good.mean():.1%}")
    print("Most common coordinates (watch for a default point):")
    print(top_pt.to_string())

{'type': 'type', 'reason': 'reason', 'opened': 'open_dt'}


Date range: 2026-01-01 to 2026-10-20  (dates parsed: 100.0%)

Food-safety-related types (edit FOOD_311 to tune):
type
Rodent Activity                                    5234
Pest Infestation - Residential                     2177
Unsanitary Conditions - Establishment               635
Unsanitary Conditions - Food                        214
Food Alert - Confirmed                              170
Food Alert - Unconfirmed                            116
Overflowing or Un-kept Dumpster                      59
Unsanitary Conditions - Employees                    42
Fire in Food Establishment                           33
Dumpster & Loading Noise Disturbances                28
No Utilities - Food Establishment - Water             8
No Utilities - Food Establishment - Flood             5
No Utilities - Food Establishment - Electricity       3
No Utilities - Food Establishment - Sewer             3
Rat Bite                                              2



Rows with usable Boston coordinates: 99.8%
Most common coordinates (watch for a default point):
lat        lon       
42.360295  -71.058270    235
42.349008  -71.082437    202
42.343695  -71.065841    198


### 311 feature: food-safety complaints near each establishment

Counts complaints within `RADIUS_M` meters in the `WINDOW_311_DAYS` days **before** each inspection.
This is the pandas prototype; the production version should live in dbt (`ST_DWITHIN` in BigQuery).

In [29]:
f["complaints_311_near"] = np.nan

if BUILD_311_FEATURES and r311 is not None:
    from sklearn.neighbors import BallTree

    EARTH_R = 6_371_000
    c = r311[
        r311[R["type"]].astype(str).str.contains(FOOD_311, case=False, regex=True)
        & in_boston(r311["lat"], r311["lon"])
        & r311["opened"].notna()
    ]
    # drop the default geocode point if one point holds >1% of complaints
    pt_counts = c.groupby(["lat", "lon"])["opened"].transform("size")
    c = c[pt_counts < max(50, 0.01 * len(c))]
    print(f"Using {len(c):,} food-safety 311 complaints")

    c_dates = c["opened"].values.astype("datetime64[D]")
    tree = BallTree(np.radians(c[["lat", "lon"]].values), metric="haversine")

    ok = in_boston(f["lat"], f["lon"])
    q = f.loc[ok, ["lat", "lon", "insp_date"]]
    counts = np.full(len(q), np.nan)
    window = np.timedelta64(WINDOW_311_DAYS, "D")
    for start in range(0, len(q), 5000):
        chunk = q.iloc[start : start + 5000]
        neighbors = tree.query_radius(np.radians(chunk[["lat", "lon"]].values), r=RADIUS_M / EARTH_R)
        dates = chunk["insp_date"].values.astype("datetime64[D]")
        for i, idx in enumerate(neighbors):
            d = c_dates[idx]
            counts[start + i] = np.count_nonzero((d < dates[i]) & (d >= dates[i] - window))
    f.loc[ok, "complaints_311_near"] = counts

    # The 311 snapshot only goes back to its first record. Before that + a full window, a 0 means
    # "no data", not "no complaints", so leave those rows missing.
    covered_from = pd.Timestamp(c_dates.min()) + pd.Timedelta(days=WINDOW_311_DAYS)
    f.loc[f["insp_date"] < covered_from, "complaints_311_near"] = np.nan
    print(f"311 feature populated from {covered_from:%Y-%m-%d}: {f['complaints_311_near'].notna().mean():.1%} of inspections")
    print(f["complaints_311_near"].describe().to_string())

    covered = f[f["is_routine"] & f["complaints_311_near"].notna()]
    if len(covered):
        print(f"\nCritical rate by nearby complaints ({len(covered):,} routine inspections with coverage):")
        display(rate_table(covered, pd.cut(covered["complaints_311_near"], [-1, 0, 2, 5, 10, 1_000]), min_n=30))
else:
    print("Skipping 311 features")

Using 8,701 food-safety 311 complaints


311 feature populated from 2026-04-01: 3.4% of inspections
count    6562.000000
mean       11.926242
std        10.626269
min         0.000000
25%         4.000000
50%         9.000000
75%        17.000000
max        74.000000

Critical rate by nearby complaints (3,745 routine inspections with coverage):


,inspections,critical_rate,lift
complaints_311_near,,,
"(-1, 0]",299,0.157,0.624
"(0, 2]",468,0.205,0.815
"(2, 5]",590,0.208,0.828
"(5, 10]",789,0.264,1.047
"(10, 1000]",1599,0.293,1.165


### Answer: 311

**Promising, but it can't be used for training yet.**

- The 311 snapshot only covers January–October 2026, so the feature exists for 3% of inspections (April 2026 on),
  all in the test window. No training rows have it, so section 11 drops it automatically.
- On the 3.6k covered routine inspections, the direction is clear. The critical rate rises from 15% with no nearby
  food-safety complaints to 30% with more than 10 (lift 0.61 → 1.18).
- Caveat: complaint counts also track density (downtown and busy commercial streets have more of everything), so
  part of this is location, not hygiene. That needs checking once there's training data.

**Action for ingestion:** backfill legacy 311 history (data.boston.gov publishes one file per year back to 2011)
to at least 2020.

## 11. Time-based split and baselines

Train on older inspections, validate on the next block, test on the most recent `TEST_MONTHS`.
Metrics that matter for a ranking problem:

- **ROC AUC / PR AUC**: overall ranking quality
- **Precision @ top 20%**: of the establishments we'd send inspectors to first, how many actually had a critical violation?
- **Lift @ top 20%**: that precision divided by the base rate

In [30]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# license_age_days is deliberately left out: it only exists for licenses still active today (section 9)
NUMERIC = [
    "n_prior_insp", "days_since_prev", "days_since_first_insp", "prev_n_critical", "prev_n_violations",
    "prev_result_fail", "last_routine_n_critical", "days_since_last_routine", "had_followup_since_last_routine",
    "prior_critical_total", "prior_violations_total", "prior_critical_rate",
    "critical_past_365d", "violations_past_365d", "month", "fda_code_era", "complaints_311_near",
]
CATEGORICAL = ["licensecat", "city"]

model_df = f[f["is_routine"]].copy()
end = model_df["insp_date"].max()
test_start = end - pd.DateOffset(months=TEST_MONTHS)
valid_start = test_start - pd.DateOffset(months=VALID_MONTHS)

train = model_df[model_df["insp_date"] < valid_start]
valid = model_df[(model_df["insp_date"] >= valid_start) & (model_df["insp_date"] < test_start)]
test = model_df[model_df["insp_date"] >= test_start]

# Only keep features the training set actually has (drops the 311 count until history is backfilled)
NUMERIC = [c for c in NUMERIC if c in f and train[c].notna().any()]
CATEGORICAL = [c for c in CATEGORICAL if c in f]

split_summary = pd.DataFrame({
    s: {"from": d["insp_date"].min().date(), "to": d["insp_date"].max().date(), "rows": len(d), "base_rate": round(d["target"].mean(), 3)}
    for s, d in [("train", train), ("  of which current era", train[train["insp_date"] >= FDA_ERA_START]), ("valid", valid), ("test", test)]
}).T
print("Features:", NUMERIC + CATEGORICAL)
split_summary

Features: ['n_prior_insp', 'days_since_prev', 'days_since_first_insp', 'prev_n_critical', 'prev_n_violations', 'prev_result_fail', 'last_routine_n_critical', 'days_since_last_routine', 'had_followup_since_last_routine', 'prior_critical_total', 'prior_violations_total', 'prior_critical_rate', 'critical_past_365d', 'violations_past_365d', 'month', 'fda_code_era', 'licensecat', 'city']


,from,to,rows,base_rate
train,2006-04-04,2024-10-18,105982,0.301
of which current era,2020-01-02,2024-10-18,31086,0.215
valid,2024-10-19,2025-10-18,7154,0.2
test,2025-10-20,2026-10-19,7626,0.238


In [31]:
def preprocessor(scale, numeric=None):
    numeric = NUMERIC if numeric is None else numeric
    num_steps = [("impute", SimpleImputer(strategy="median", add_indicator=True))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    transformers = [("num", Pipeline(num_steps), numeric)]
    if CATEGORICAL:
        cat = Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20)),
        ])
        transformers.append(("cat", cat, CATEGORICAL))
    return ColumnTransformer(transformers, sparse_threshold=0)


def x(df, numeric=None):
    numeric = NUMERIC if numeric is None else numeric
    out = df[numeric + CATEGORICAL].copy()
    for c in CATEGORICAL:
        out[c] = out[c].astype(object).where(out[c].notna(), "missing").astype(str)
    return out


def evaluate(name, y, score, top=0.20):
    k = max(1, int(len(y) * top))
    top_idx = np.argsort(-np.asarray(score))[:k]
    p_at_k = np.asarray(y)[top_idx].mean()
    return {
        "model": name,
        "roc_auc": roc_auc_score(y, score),
        "pr_auc": average_precision_score(y, score),
        f"precision@{int(top * 100)}%": p_at_k,
        f"lift@{int(top * 100)}%": p_at_k / np.mean(y),
    }


def make_models():
    return {
        "logistic_regression": Pipeline([("prep", preprocessor(scale=True)), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))]),
        "hist_gradient_boosting": Pipeline([("prep", preprocessor(scale=False)), ("clf", HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, early_stopping=True, random_state=882))]),
    }

**How to handle the 2019 code change:** train on all years and ignore it, train on all years with a flag for the
code era, or drop the old era entirely?

In [32]:
BASE_NUMERIC = [c for c in NUMERIC if c != "fda_code_era"]
window_check = []
for label, tr, num in [
    ("all years, no era flag", train, BASE_NUMERIC),
    ("all years + era flag (used below)", train, NUMERIC),
    (f"current era only (since {FDA_ERA_START})", train[train["insp_date"] >= FDA_ERA_START], BASE_NUMERIC),
]:
    pipe = Pipeline([
        ("prep", preprocessor(scale=False, numeric=num)),
        ("clf", HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, early_stopping=True, random_state=882)),
    ]).fit(x(tr, num), tr["target"])
    score = pipe.predict_proba(x(valid, num))[:, 1]
    window_check.append({"train window": label, "train rows": len(tr), **evaluate("hist_gradient_boosting", valid["target"], score),
                         "mean predicted": score.mean(), "valid base rate": valid["target"].mean()})
pd.DataFrame(window_check).set_index("train window").drop(columns="model").round(3)

,train rows,roc_auc,pr_auc,precision@20%,lift@20%,mean predicted,valid base rate
train window,,,,,,,
"all years, no era flag",105982,0.734,0.416,0.436,2.173,0.250,0.2
all years + era flag (used below),105982,0.739,0.409,0.427,2.128,0.207,0.2
current era only (since 2020-01-01),31086,0.741,0.410,0.431,2.149,0.206,0.2


In [33]:
models = make_models()
results = []
for split_name, split in [("valid", valid), ("test", test)]:
    # Rule-of-thumb baselines: rank by past critical violations
    results.append({"split": split_name, **evaluate("rule: last_routine_n_critical", split["target"], split["last_routine_n_critical"].fillna(0))})
    results.append({"split": split_name, **evaluate("rule: critical_past_365d", split["target"], split["critical_past_365d"].fillna(0))})

for name, pipe in models.items():
    pipe.fit(x(train), train["target"])
    for split_name, split in [("valid", valid), ("test", test)]:
        results.append({"split": split_name, **evaluate(name, split["target"], pipe.predict_proba(x(split))[:, 1])})

pd.DataFrame(results).set_index(["split", "model"]).round(3)

roc_auc  pr_auc  precision@20%  lift@20%
split model                                                                  
valid rule: last_routine_n_critical    0.632   0.301          0.408     2.037
      rule: critical_past_365d         0.647   0.310          0.383     1.908
test  rule: last_routine_n_critical    0.600   0.307          0.411     1.725
      rule: critical_past_365d         0.602   0.307          0.373     1.565
valid logistic_regression              0.715   0.382          0.400     1.996
test  logistic_regression              0.687   0.400          0.435     1.824
valid hist_gradient_boosting           0.739   0.409          0.427     2.128
test  hist_gradient_boosting           0.719   0.437          0.457     1.917

**How to read this:** the models need to clearly beat the rule-of-thumb rows, or the extra features aren't
pulling their weight. Report test numbers only once you're done tuning on validation.

In [34]:
from sklearn.inspection import permutation_importance

best = models["hist_gradient_boosting"]
imp = permutation_importance(best, x(valid), valid["target"], scoring="average_precision", n_repeats=5, random_state=882)
importance = pd.Series(imp.importances_mean, index=NUMERIC + CATEGORICAL).sort_values()
ax = importance.plot.barh(figsize=(8, 5), title="Permutation importance (validation, PR AUC drop)")
plt.show()
print(importance.sort_values(ascending=False).round(4).to_string())

last_routine_n_critical            0.0325
prior_violations_total             0.0240
days_since_first_insp              0.0231
days_since_prev                    0.0117
violations_past_365d               0.0109
prev_n_violations                  0.0103
prior_critical_rate                0.0055
critical_past_365d                 0.0051
prev_n_critical                    0.0041
days_since_last_routine            0.0037
city                               0.0025
prev_result_fail                   0.0015
licensecat                         0.0012
n_prior_insp                       0.0005
prior_critical_total               0.0003
fda_code_era                       0.0000
month                             -0.0013
had_followup_since_last_routine   -0.0034


### Answer: baseline

**Supported: history features rank risk well beyond chance and beat the rules of thumb.**

| | Valid ROC AUC | Valid precision@20% | Test ROC AUC | Test precision@20% |
|---|---|---|---|---|
| Best rule (`last_routine_n_critical`) | 0.63 | 40.8% | 0.60 | 41.2% |
| Logistic regression | 0.72 | 40.3% | 0.69 | 43.6% |
| Gradient boosting | **0.74** | **43.8%** | **0.72** | **45.7%** |

- If inspectors visited the model's top 20% first, ~44–45% of those visits would find a critical violation,
  about **2× the base rate**. That's a few points better than the best rule, with much better ranking across the
  whole list (+0.1 ROC AUC).
- **The 2019 code change:** all three options rank about equally well (validation ROC AUC 0.738–0.742), so the
  choice is about calibration. Ignoring the shift over-predicts (mean prediction 24.8% vs 20.1% actual). Both the
  era flag and dropping pre-2020 data are calibrated (20.7%). The flag keeps the 3× larger training set and has
  the best AUC, so it's the default. Its zero permutation importance is only because it's constant within the
  validation year.
- **Top features:** criticals at the last routine inspection, lifetime violation count, establishment age
  (`days_since_first_insp`), days since the previous visit, and violations in the past year. `month`, `city` and
  `had_followup_since_last_routine` contribute ~nothing.
- **Watch for drift.** The test-year base rate (23.8%) is higher than validation (20.1%), part of a 2026 uptick
  also visible in section 6.

## 12. Save the feature table and next steps

Saves to `data/` (gitignored). The dbt feature mart should eventually reproduce this table.

**Next steps**
- [ ] Record the latest inspection date, row counts and real column names in `docs/data_sources.md`
- [ ] Hand Kai the cleaning table from section 2 for `stg_food_inspections.sql`
- [ ] Ingestion: backfill legacy 311 history to 2020+ so the complaints feature can be trained
- [ ] Port the inspection grain + features into dbt (`int_inspections`, `fct_inspection_features`)
- [ ] Replace the `city` field with a spatial join to the `neighborhoods` GeoJSON
- [ ] Robustness check: `**`-or-`***` target; follow-up window of 21 vs 30 vs 45 days
- [ ] Notebook 02: Chicago-style evaluation (how many days sooner are critical violations found?)
- [ ] MLflow tracking and hyperparameter tuning

In [35]:
OUT_DIR = os.getenv("FEATURE_OUT_DIR", "../../data")
os.makedirs(OUT_DIR, exist_ok=True)
out_path = os.path.join(OUT_DIR, "features_v0.parquet")
keep = ["license", "insp_date", "target", "result_fail", "is_routine", "lat", "lon"] + NUMERIC + CATEGORICAL
f[keep].to_parquet(out_path, index=False)
print(f"Saved {len(f):,} rows to {out_path}")

Saved 195,653 rows to ../../data\features_v0.parquet
